# 03 — Retrieval Demo

Full pipeline: **PDF → extract → classify → chunk → embed → Qdrant + MongoDB → retrieve**

This notebook demonstrates the Week 3 storage & retrieval layer:
1. Ingest 2 PDFs (Java.pdf, C.pdf) into MongoDB + Qdrant
2. Query 5 ground-truth questions and inspect retrieved chunks

In [1]:
import logging
import os
from pathlib import Path

from dotenv import load_dotenv

load_dotenv()  # loads .env from project root

logging.basicConfig(level=logging.INFO, format="%(levelname)s | %(name)s | %(message)s")

DATA_DIR = Path("../data")
print("PDFs available:", [f.name for f in DATA_DIR.glob("*.pdf")])
print("GOOGLE_API_KEY set:", bool(os.environ.get("GOOGLE_API_KEY")))
print("QDRANT_URL set:", bool(os.environ.get("QDRANT_URL")))
print("MONGO_URI set:", bool(os.environ.get("MONGO_URI", "default")))

PDFs available: ['C.pdf', 'Java.pdf', 'Python.pdf']
GOOGLE_API_KEY set: True
QDRANT_URL set: True
MONGO_URI set: True


## 1. Ingest PDFs

Run the full pipeline for each PDF: extract → classify → chunk → embed → store.

In [2]:
from hsc_edu.storage.ingest import ingest_pdf

n_java = ingest_pdf(DATA_DIR / "Java.pdf", subject="Lập trình Java", doc_id="java")
print(f"Ingested {n_java} chunks from Java.pdf")

INFO | hsc_edu.storage.ingest | Ingesting 'Java.pdf' (subject='Lập trình Java', doc_id='java')…
INFO | hsc_edu.core.extraction.pdf_detector | PDF detection: Java.pdf → text-based  (text_pages=231, scan_pages=10, ratio=0.96)
INFO | hsc_edu.core.extraction.text_extractor | Extracted 2121 blocks from 'Java.pdf' (2363 raw → 2121 after noise filter)
INFO | hsc_edu.core.classification.block_classifier | Loaded classification config from C:\Homework\Code File\Python Code File\de an\hsc_edu\config\subject_configs\default.yaml
INFO | hsc_edu.core.classification.block_classifier | Classified 2121 blocks: 210 headings, 1897 paragraphs, 14 special
INFO | hsc_edu.core.chunking.text_chunker | Chunked 2121 blocks → 211 groups → 322 raw chunks → 322 final chunks
INFO | httpx | HTTP Request: GET https://b789f557-a87e-425a-98ac-ce034dd4c6a0.us-east-1-1.aws.cloud.qdrant.io:6333/collections/hsc_edu_chunks/exists "HTTP/1.1 200 OK"
INFO | httpx | HTTP Request: GET https://b789f557-a87e-425a-98ac-ce034dd4c6a

ClientError: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/embed_content_free_tier_requests, limit: 1000, model: gemini-embedding-1.0\nPlease retry in 30.160749009s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/embed_content_free_tier_requests', 'quotaId': 'EmbedContentRequestsPerDayPerUserPerProjectPerModel-FreeTier', 'quotaDimensions': {'location': 'global', 'model': 'gemini-e

In [ ]:
n_c = ingest_pdf(DATA_DIR / "C.pdf", subject="Lập trình C", doc_id="c-lang")
print(f"Ingested {n_c} chunks from C.pdf")

## 2. Storage stats

In [ ]:
from hsc_edu.storage.mongo_store import MongoChunkStore
from hsc_edu.storage.vector_store import QdrantVectorStore

mongo = MongoChunkStore()
qdrant = QdrantVectorStore()

print(f"MongoDB total chunks : {mongo.count()}")
print(f"MongoDB subjects     : {mongo.distinct_values('subject')}")
print(f"MongoDB doc_ids      : {mongo.distinct_values('doc_id')}")
print()
print(f"Qdrant collection    : {qdrant.collection_info()}")

## 3. Retrieval tests — ground truth questions

We test 5 questions from `data/note/ground_truth.md` to check if the
retrieved chunks contain the expected content.

In [ ]:
from hsc_edu.storage.retrieval import retrieve_chunks

def show_results(query: str, results: list, max_preview: int = 200):
    print(f"\n{'=' * 80}")
    print(f"QUERY: {query}")
    print(f"{'=' * 80}")
    if not results:
        print("  (no results)")
        return
    for i, (chunk, score) in enumerate(results):
        path_str = " > ".join(chunk.section_path) if chunk.section_path else "(no heading)"
        preview = chunk.text[:max_preview].replace("\n", " ↵ ")
        if len(chunk.text) > max_preview:
            preview += " …"
        print(f"\n  [{i+1}] score={score:.4f}  │  {chunk.subject}  │  pages {chunk.page_start}–{chunk.page_end}")
        print(f"      Section: {path_str}")
        print(f"      Text   : {preview}")

In [ ]:
# Q1 (Java) — Bốn nguyên tắc OOP
q1 = "Bốn nguyên tắc trụ cột của lập trình hướng đối tượng là gì?"
r1 = retrieve_chunks(q1, subject="Lập trình Java", top_k=5)
show_results(q1, r1)

In [ ]:
# Q2 (Java) — Tính đa hình
q2 = "Tính đa hình trong lập trình hướng đối tượng được hiểu như thế nào?"
r2 = retrieve_chunks(q2, subject="Lập trình Java", top_k=5)
show_results(q2, r2)

In [ ]:
# Q3 (C) — Thuật giải là gì
q3 = "Thuật giải (Algorithm) là gì và nó dựa trên những cấu trúc điều khiển cơ bản nào?"
r3 = retrieve_chunks(q3, subject="Lập trình C", top_k=5)
show_results(q3, r3)

In [ ]:
# Q4 (C) — Kiểu dữ liệu cơ bản
q4 = "Liệt kê các kiểu dữ liệu cơ bản trong C và kích thước bộ nhớ tương ứng?"
r4 = retrieve_chunks(q4, subject="Lập trình C", top_k=5)
show_results(q4, r4)

In [ ]:
# Q5 (cross-subject, no filter) — Thừa kế
q5 = "Thế nào là quan hệ thừa kế giữa lớp cha và lớp con?"
r5 = retrieve_chunks(q5, top_k=5)
show_results(q5, r5)

## 4. Summary

| # | Question | Subject filter | Expected topic | Relevant? |
|---|----------|---------------|----------------|------------|
| 1 | 4 nguyên tắc OOP | Java | OOP principles | |
| 2 | Đa hình | Java | Polymorphism | |
| 3 | Thuật giải | C | Algorithm definition | |
| 4 | Kiểu dữ liệu | C | Data types | |
| 5 | Thừa kế | (none) | Inheritance | |

Fill in the **Relevant?** column after inspecting the results above.